# MirrorMate - Chess Move Predictor

**Trained on my own Chess.com games** using a Convolutional Neural Network.

This notebook downloads my games, converts them into training data, trains a CNN (ChessNet), and provides an interactive board where the model predicts the next move in my playing style.

**Team:** Mohamed Thoufic, Arun Neupane, Nistal Gigi Thomas  
**Subject:** Deep Learning (Summer Semester Project)

## 1. Setup
Install required libraries and connect to Google Drive to save the trained model.

In [ ]:
!pip install python-chess requests torch torchvision torchaudio tqdm numpy pandas -q

from google.colab import drive
import os

drive.mount('/content/drive', force_remount=True)

MODEL_DIR = "/content/drive/MyDrive/ChessBot_Project"
os.makedirs(MODEL_DIR, exist_ok=True)
MODEL_PATH = os.path.join(MODEL_DIR, "chess_model.pth")

print("✅ Setup complete")

## 2. Download My Games from Chess.com
Uses the Chess.com public API to download recent games of the hardcoded username.

In [ ]:
import requests
import time

YOUR_EXACT_USERNAME = "mohamedthoufic"
username = YOUR_EXACT_USERNAME.strip().lower()
headers = {"User-Agent": "MirrorMateBot/1.0"}

archive_url = f"https://api.chess.com/pub/player/{username}/games/archives"
response = requests.get(archive_url, headers=headers)
archives = response.json().get("archives", []) if response.status_code == 200 else []

games_data = []
for url in archives[-4:]:  # last 4 months
    res = requests.get(url, headers=headers)
    if res.status_code == 200:
        games_data.extend(res.json().get("games", []))
    time.sleep(0.5)

print(f"✅ Loaded {len(games_data)} games")

## 3. Convert Games into Training Data
Every board position is converted into a 12×8×8 matrix (6 piece types × 2 colors).
We also record the from-square and to-square of the move that was actually played.

In [ ]:
import numpy as np
import chess
import chess.pgn
import io

def board_to_matrix(board):
    """Convert a chess.Board into a 12×8×8 float32 tensor."""
    matrix = np.zeros((12, 8, 8), dtype=np.float32)
    piece_map = {
        chess.PAWN: 0, chess.KNIGHT: 1, chess.BISHOP: 2,
        chess.ROOK: 3, chess.QUEEN: 4, chess.KING: 5
    }
    for sq, piece in board.piece_map().items():
        row, col = divmod(sq, 8)
        ch = piece_map[piece.piece_type] + (6 if piece.color == chess.BLACK else 0)
        matrix[ch, row, col] = 1.0
    return matrix

X, y_from, y_to = [], [], []

for game_obj in games_data:
    pgn_text = game_obj.get("pgn", "")
    if not pgn_text:
        continue
    game = chess.pgn.read_game(io.StringIO(pgn_text))
    if not game:
        continue

    board = game.board()
    for move in game.mainline_moves():
        X.append(board_to_matrix(board))
        y_from.append(move.from_square)
        y_to.append(move.to_square)
        board.push(move)

X = np.array(X)
y_from = np.array(y_from)
y_to = np.array(y_to)

print(f"✅ Processed {len(X)} positions")

## 4. Define and Train ChessNet
A simple CNN with two output heads:
- one predicts the starting square
- one predicts the destination square

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Device: {device}")

class ChessNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(12, 64, 3, padding=1), nn.ReLU(),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(),
            nn.Conv2d(128, 128, 3, padding=1), nn.ReLU(),
            nn.Flatten()
        )
        self.fc_from = nn.Linear(128 * 8 * 8, 64)
        self.fc_to   = nn.Linear(128 * 8 * 8, 64)

    def forward(self, x):
        f = self.conv(x)
        return self.fc_from(f), self.fc_to(f)

model = ChessNet().to(device)

# Load existing model if available, otherwise train
if os.path.exists(MODEL_PATH) and len(X) < 1000:
    model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
    print("✅ Loaded saved model")
else:
    print("🆕 Training model...")
    if len(X) == 0:
        raise ValueError("Run the data cells first!")

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    dataset = TensorDataset(
        torch.tensor(X, dtype=torch.float32),
        torch.tensor(y_from, dtype=torch.long),
        torch.tensor(y_to, dtype=torch.long)
    )
    loader = DataLoader(dataset, batch_size=128, shuffle=True)

    model.train()
    epochs = 8

    for epoch in range(epochs):
        total_loss = 0
        for bx, byf, byt in loader:
            bx, byf, byt = bx.to(device), byf.to(device), byt.to(device)
            optimizer.zero_grad()
            of, ot = model(bx)
            loss = criterion(of, byf) + criterion(ot, byt)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch {epoch+1}/{epochs}  Loss: {total_loss/len(loader):.4f}")

    torch.save(model.state_dict(), MODEL_PATH)
    print("✅ Training complete & model saved")

model.eval()

## 5. Prediction Backend
Registers a callback that the interactive HTML board can call.
It only considers legal moves and picks the one with highest P(from) × P(to).

In [ ]:
from google.colab import output
import json

def predict_move_callback(fen_string):
    try:
        board = chess.Board(fen_string)
        legal_moves = list(board.legal_moves)
        if not legal_moves:
            return json.dumps({"success": False, "error": "No legal moves"})

        tensor = torch.from_numpy(board_to_matrix(board)).unsqueeze(0).to(device).float()
        with torch.no_grad():
            out_from, out_to = model(tensor)

        pf = torch.softmax(out_from, dim=1).cpu().numpy()[0]
        pt = torch.softmax(out_to, dim=1).cpu().numpy()[0]

        best_move = max(legal_moves, key=lambda m: pf[m.from_square] * pt[m.to_square])
        return json.dumps({"success": True, "move": best_move.uci()})
    except Exception as e:
        return json.dumps({"success": False, "error": str(e)})

output.register_callback('notebook.predict_move', predict_move_callback)
print("✅ Backend ready")

## 6. Interactive Frontend
Dark-themed chessboard with:
- Drag & drop pieces
- White / Black turn selector
- Predict Next Move button
- Manual move input
- Clear / Reset buttons

In [ ]:
from IPython.display import HTML, display

html_code = """
<!DOCTYPE html>
<html>
<head>
<link rel="stylesheet" href="https://unpkg.com/@chrisoakman/chessboardjs@1.0.0/dist/chessboard-1.0.0.min.css">
<script src="https://code.jquery.com/jquery-3.5.1.min.js"></script>
<script src="https://unpkg.com/@chrisoakman/chessboardjs@1.0.0/dist/chessboard-1.0.0.min.js"></script>
</head>
<body>
<div style="display:flex;flex-direction:column;align-items:center;justify-content:center;font-family:'Segoe UI',sans-serif;padding:20px;background:#262522;border-radius:12px;max-width:420px;margin:auto;color:#fff;">
  <h3 style="margin-top:0;color:#f1f1f1;">MirrorMate Predictor</h3>
  <div id="chessboard_frame" style="width:360px;margin-bottom:15px;border:2px solid #403e3b;border-radius:4px;overflow:hidden;"></div>

  <div style="display:flex;gap:10px;margin-bottom:12px;width:100%;">
    <select id="turn_modifier" style="flex:1;padding:10px;border-radius:6px;border:1px solid #403e3b;background:#312e2b;color:#bababa;font-weight:bold;">
      <option value="w">White to Move</option>
      <option value="b">Black to Move</option>
    </select>
    <button id="predict_btn" style="flex:1;background:#81b64c;color:white;border:none;padding:10px;border-radius:6px;cursor:pointer;font-weight:bold;">Predict Next Move</button>
  </div>

  <div style="width:100%;margin-bottom:12px;">
    <input type="text" id="manual_move" placeholder="Type move e.g. e2e4" style="width:100%;padding:10px;border-radius:6px;border:1px solid #403e3b;background:#312e2b;color:#bababa;">
    <button id="apply_btn" style="width:100%;margin-top:8px;background:#4a90e2;color:white;border:none;padding:10px;border-radius:6px;cursor:pointer;font-weight:bold;">Apply Move & Predict Response</button>
  </div>

  <div style="display:flex;gap:10px;width:100%;">
    <button id="clear_btn" style="flex:1;background:#312e2b;color:#bababa;border:1px solid #403e3b;padding:8px;border-radius:6px;cursor:pointer;">Clear</button>
    <button id="reset_btn" style="flex:1;background:#312e2b;color:#bababa;border:1px solid #403e3b;padding:8px;border-radius:6px;cursor:pointer;">Reset Start</button>
  </div>
  <div id="status" style="width:100%;text-align:center;font-size:13px;color:#989795;margin-top:10px;">Ready</div>
</div>

<script>
var board = Chessboard('chessboard_frame', {
  draggable: true,
  dropOffBoard: 'trash',
  sparePieces: true,
  position: 'start',
  pieceTheme: 'https://koblenski.github.io/javascript/chessboardjs-0.3.0/img/chesspieces/wikipedia/{piece}.png'
});

function predict() {
  $('#status').text('Thinking...');
  var fen = board.fen() + ' ' + $('#turn_modifier').val() + ' KQkq - 0 1';
  google.colab.kernel.invokeFunction('notebook.predict_move', [fen], {}).then(r => {
    var p = JSON.parse(r.data['text/plain']);
    if (p.success) {
      board.move(p.move.substring(0,2) + '-' + p.move.substring(2,4));
      $('#status').text('✅ ' + p.move);
    } else {
      $('#status').text('❌ ' + p.error);
    }
  }).catch(() => $('#status').text('❌ Communication error - try again'));
}

$('#predict_btn').on('click', predict);

$('#apply_btn').on('click', function() {
  var m = $('#manual_move').val().trim();
  if (m && board.move(m)) {
    $('#status').text('Applied ' + m);
    setTimeout(predict, 400);
  } else {
    $('#status').text('Invalid move');
  }
});

$('#clear_btn').on('click', () => { board.clear(); $('#status').text('Cleared'); });
$('#reset_btn').on('click', () => { board.start(); $('#status').text('Reset'); });
</script>
</body>
</html>
"""

display(HTML(html_code))